<h1 style="color:#5DADE2; text-align:center;">🩺 Smart Detection of Diabetic Foot Ulcers</h1>

<hr>

<h2 style="color:#58D68D;">📌 Introduction</h2>

<p style="font-size:18px; color:#EAECEE;">
A diabetic foot ulcer is a type of wound that appears on the feet of people with diabetes. 
It mainly occurs due to poor blood circulation and nerve damage (neuropathy), which reduce the ability to feel pain and slow down the healing process.
</p>

<p style="font-size:18px; color:#EAECEE;">
As a result, even small cuts, injuries, or pressure on the foot can develop into serious ulcers if not treated properly.
</p>

<h2 style="color:#58D68D;">⚠️ Risks and Complications</h2>

<p style="font-size:18px; color:#D5D8DC;">
Diabetic foot ulcers are considered a serious complication because they can easily become infected. 
In severe cases, they may lead to tissue damage or even amputation if medical care is delayed.
</p>

<h2 style="color:#58D68D;">✅ Prevention</h2>

<ul style="font-size:18px; color:#D5D8DC;">
  <li>✔️ Maintaining good blood sugar control</li>
  <li>✔️ Regular foot inspection</li>
  <li>✔️ Early medical intervention</li>
</ul>

<hr>

<p style="text-align:center; font-size:16px; color:#ABB2B9;">
✨ This project aims to detect diabetic foot ulcers using machine learning techniques.
</p>

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image 
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import tensorflow as tf
from tensorflow import keras 
from tensorflow.keras import losses
from tensorflow.keras import callbacks
from tensorflow.keras import layers
from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam , Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.layers import Conv2D, MaxPooling2D , GlobalAveragePooling2D,Dropout,BatchNormalization,Input,Flatten,Dense
from tensorflow.keras.applications import EfficientNetB0




2026-05-04 13:37:49.693361: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777901870.057757      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777901870.158318      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777901871.049031      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777901871.049071      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777901871.049074      57 computation_placer.cc:177] computation placer alr

In [ ]:
image_path = '/kaggle/input/datasets/laithjj/diabetic-foot-ulcer-dfu/DFU'

for root, dirs, files in os.walk(image_path):
    print("Folder:", root)
    print("Number of images:", len(files))
    

In [ ]:
import os

image_path = '/kaggle/input/datasets/laithjj/diabetic-foot-ulcer-dfu/DFU'

def get_folders_with_images(path):
    folders_info = []
    for root, dirs, files in os.walk(path):
        # لو فيه صور في الفولدر
        num_images = len([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if num_images > 0:
            folders_info.append((root, num_images))
    return folders_info

folders_info = get_folders_with_images(image_path)

for folder, count in folders_info:
    print(f"Folder: {folder}")
    print(f"Number of images: {count}\n")

print(f"Total number of classes (folders with images): {len(folders_info)}")

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

# مسار الداتا الرئيسي
image_path = '/kaggle/input/datasets/laithjj/diabetic-foot-ulcer-dfu/DFU'

# نجيب كل الفولدرات اللي فيها صور
folders_with_images = []
for root, dirs, files in os.walk(image_path):
    # فلترة الملفات: ناخد بس الصور
    img_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if len(img_files) > 0:
        folders_with_images.append((root, img_files))

# نطبع عدد الصور في كل فولدر
for folder, imgs in folders_with_images:
    print(f"Folder: {folder}")
    print(f"Number of images: {len(imgs)}\n")

# نعرض صورة عشوائية من كل فولدر
for folder, imgs in folders_with_images:
    if len(imgs) == 0:  # لو الفولدر فاضي بالخطأ، نتخطاه
        continue
    img_name = random.choice(imgs)
    img_path = os.path.join(folder, img_name)
    img = image.load_img(img_path)
    
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(folder))
    plt.show()

In [ ]:
import os

image_path = '/kaggle/input/datasets/laithjj/diabetic-foot-ulcer-dfu/DFU'

classes = [f for f in os.listdir(image_path) if os.path.isdir(os.path.join(image_path, f))]
print("Number of classes:", len(classes))
print("Class names:", classes)

In [ ]:
patch_path = os.path.join(image_path, 'Patches')
classes = [f for f in os.listdir(patch_path) if os.path.isdir(os.path.join(patch_path, f))]
print("Number of real classes:", len(classes))
print("Class names:", classes)

In [ ]:
TrainPath = '/kaggle/input/datasets/laithjj/diabetic-foot-ulcer-dfu/DFU/Patches'
TestPath  = '/kaggle/input/datasets/laithjj/diabetic-foot-ulcer-dfu/DFU/TestSet'

batchsize    = 32
image_height = 224
image_width = 224
num_classes  = 2 # Normal and Abnormal
epochs       = 200

In [ ]:
train_data = image_dataset_from_directory(
    TrainPath,
    labels = 'inferred', # Abnormal(Ulcer) | Normal(Healthy skin) 
    validation_split = 0.2,
    batch_size = batchsize,
    image_size = (image_width,image_height),
    subset ='training',
    seed = 0
)


validation_ds = image_dataset_from_directory(
    TrainPath,
    labels = 'inferred', # Abnormal(Ulcer) | Normal(Healthy skin) 
    validation_split = 0.2,
    batch_size = batchsize,
    image_size = (image_width,image_height),
    subset ='validation',
    seed = 0
)


In [ ]:
train_data

In [ ]:
for images, labels in train_data.take(1):# take first batch
        print("Images shape: ", images.shape)
        print("Labels shape: ", labels.shape)
        print("Labels : ", labels.numpy())

In [ ]:
for images, labels in train_data.take(1):
    plt.figure(figsize=(10,10))
    for i in range(9):
           ax = plt.subplot(3,3,i+1)
           plt.imshow(images[i].numpy().astype("uint8"))
           plt.title(f"label: {labels[i].numpy()}")
           plt.axis("off")
           
    plt.show()

In [ ]:
class_names = train_data.class_names
print(class_names)

In [ ]:
def decode_img(img):
    img = tf.io.decode_jpeg(img,channels=3)
    return tf.image.resize(img,[image_width,image_height])

def process_path(file_path):
    img = tf.io.read_file(file_path)# encode image
    img = decode_img(img)
    return img

test_ds = tf.data.Dataset.list_files(str(TestPath + '/*'),shuffle = False)
test_ds = test_ds.map(process_path, num_parallel_calls = tf.data.AUTOTUNE)


## Visualize the Testing Dataset

In [ ]:
plt.figure(figsize=(10,10))

i=0
for images in test_ds.take(9):
    ax = plt.subplot(3,3,i+1)
    plt.imshow(images.numpy().astype("uint8"))
    plt.axis('off')
    i+=1


In [ ]:

labels = []

for _, y in train_data:
    labels.extend(y.numpy())

labels = np.array(labels)

plt.figure()
plt.hist(labels)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

In [ ]:
class_names = train_data.class_names

plt.figure(figsize=(10,5))

for images, labels in train_data.take(1):
    for i in range(6):
        ax = plt.subplot(2,3,i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i].numpy()])
        plt.axis('off')

# Data Augmentation


In [ ]:
img_augmentation = Sequential(
    [
        layers.RandomRotation(factor = 0.15),
        layers.RandomTranslation(height_factor = 0.1,width_factor = 0.1),
        layers.RandomFlip(),
        layers.GaussianNoise(stddev =0.09),
        layers.RandomContrast(factor =0.1)
    ],
    name = 'img_augmentation'
)

# View the Augmentation


In [ ]:
plt.figure(figsize=(10,10))

for image,labels in train_data.take(1):
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        aug_img = img_augmentation(tf.expand_dims(image[0],axis=0)) # (224,224,3) --->(1,224,224,3)
        plt.imshow(aug_img[0].numpy().astype('uint8'))
        plt.title(class_names[labels[0]])
        plt.axis('off')


In [ ]:
plt.figure(figsize=(10,10))

for image,labels in train_data.take(1):
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        aug_img = img_augmentation(tf.expand_dims(image[0],axis=0)) # (224,224,3) --->(1,224,224,3)
        plt.imshow(aug_img[0].numpy().astype('uint8'))
        plt.title(class_names[labels[0]])
        plt.axis('off')


In [ ]:
plt.figure(figsize=(10,10))

for image,labels in train_data.take(1):
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        aug_img = img_augmentation(tf.expand_dims(image[0],axis=0)) # (224,224,3) --->(1,224,224,3)
        plt.imshow(aug_img[0].numpy().astype('uint8'))
        plt.title(class_names[labels[0]])
        plt.axis('off')


In [ ]:
plt.figure(figsize=(12,12))

for images, labels in train_data.take(1):
    for i in range(3):
        # الصورة الأصلية
        ax = plt.subplot(3,2,2*i+1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title("Original")
        plt.axis('off')

        # الصورة بعد augmentation
        ax = plt.subplot(3,2,2*i+2)
        aug_img = img_augmentation(tf.expand_dims(images[i], axis=0))
        plt.imshow(aug_img[0].numpy().astype('uint8'))
        plt.title("Augmented")
        plt.axis('off')

plt.suptitle("Before vs After Augmentation", fontsize=16)
plt.show()

# Catching & Prefetching (Optimizing)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.cache().prefetch(buffer_size=AUTOTUNE)
validatio_ds = validation_ds.cache().prefetch(buffer_size=AUTOTUNE)


# Define Callbacks


In [ ]:
lr_callback   = callbacks.ReduceLROnPlateau(monitor= 'val_accuracy',factor = 0.1,patience=5)
stop_callback = callbacks.EarlyStopping(monitor= 'val_accuracy',patience=8)
model_ckpt = callbacks.ModelCheckpoint("best_pretrained_model.h5",save_best_only=True,monitor="val_accuracy",mode="max")


In [ ]:
def build_model():
    inputs = layers.Input(shape=(image_width,image_height,3))
    x      = img_augmentation(inputs)

    model  = EfficientNetB0(include_top =False , weights ='imagenet',input_tensor =x)

    # Rebuild_top
    x = layers.GlobalAveragePooling2D(name='avg_pool')(model.output)
    x = layers.BatchNormalization()(x)
    
    top_dropout_rate = 0.4
    x      = layers.Dropout(top_dropout_rate,name = 'top_dropout')(x)
    output = layers.Dense(num_classes,activation = 'sigmoid',name ="pred")(x)

    # compile
    model     = tf.keras.Model(inputs,output,name='EfficientNetB0')
    optimizer = optimizers.Adam(learning_rate = 1e-3)
    loss      = losses.SparseCategoricalCrossentropy()
    model.compile(
        optimizer = optimizer, loss = loss, metrics = ['accuracy']
    )
    model.summary()
    return model


In [ ]:
model = build_model()

In [ ]:
model.fit(train_data, validation_data=validation_ds, epochs=10)

In [ ]:
model.save("model.h5")